# zynarai v1 — ACE-Step 5Hz LM Planner LoRA (Colab Pro)

Fine-tune the **5Hz LM planner** (Qwen3 **0.6B**) to transfer composition/groove
character from the 57-track core — the half the DiT LoRA can't touch. The DiT LoRA
gave you the *sound* (sub-bass, production); this gives you the *arrangement*.

Pipeline: load base model → extract `<|audio_code_N|>` targets from ~30s chunks →
SFT the Qwen3 LM with a LoRA to predict codes from the caption.

**Two honest caveats baked into the config:** (1) 57 tracks is thin for LM SFT, so we
chunk to ~30s (~350-450 examples) and use the smallest LM; (2) the prompt format must
match inference exactly — a round-trip check cell verifies this before full training.

In [ ]:
# 1 — GPU + deps. Reuses the ACE-Step env; adds PEFT/TRL for standard causal-LM SFT.
!nvidia-smi --query-gpu=name,memory.total --format=csv
%cd /content
![ -d ACE-Step-1.5 ] || git clone --depth 1 https://github.com/ace-step/ACE-Step-1.5.git
%cd /content/ACE-Step-1.5
lines = open('requirements.txt').read().splitlines()
SKIP = ('torch==','torchvision==','torchaudio==','flash-attn','--extra-index-url','triton')
open('/tmp/req.txt','w').write('\n'.join(l for l in lines if not any(t in l for t in SKIP)))
!pip install -q -r /tmp/req.txt
!pip install -q loguru lycoris-lora vector_quantize_pytorch
!pip install -q 'peft>=0.18' 'trl>=0.12' 'datasets>=2.19'
print('deps ready')

In [ ]:
# 1b — meta-tensor patch (same as the DiT notebook; base model loads the audio tokenizer)
import os, re, vector_quantize_pytorch
vq = os.path.dirname(vector_quantize_pytorch.__file__)
for fn in os.listdir(vq):
    if fn.endswith('.py'):
        p = os.path.join(vq, fn); t = open(p).read()
        t2 = re.sub(r'assert \(([^)]+?) > 1\)\.all\(\)',
                    r"assert \1.device.type == 'meta' or (\1 > 1).all()", t)
        if t2 != t: open(p,'w').write(t2); print('patched', fn)
print('vq patched')

In [ ]:
# 2 — Drive + paths (same symlink trick for the space in 'lora fine tune')
from google.colab import drive; drive.mount('/content/drive')
import os
from pathlib import Path
if not os.path.exists('/content/loradrive'):
    os.symlink('/content/drive/MyDrive/lora fine tune', '/content/loradrive')
DRIVE   = Path('/content/loradrive')
DATASET = DRIVE / 'dataset'
SFT_JSONL = DRIVE / 'lm_sft' / 'sft.jsonl'          # extracted training data (cached)
LM_OUT    = DRIVE / 'output' / 'lm_lora_0.6b'       # LM adapter checkpoints
for p in (SFT_JSONL.parent, LM_OUT): p.mkdir(parents=True, exist_ok=True)
assert (DATASET/'dataset.json').exists(), f'upload dataset to {DATASET}'
print('dataset:', DATASET, '| examples cache:', SFT_JSONL)

In [ ]:
# 3 — Checkpoints + copy the audio + our prep helpers
CKPT='/content/checkpoints'
from huggingface_hub import snapshot_download
snapshot_download('ACE-Step/Ace-Step1.5', local_dir=CKPT)
snapshot_download('ACE-Step/acestep-v15-base', local_dir=f'{CKPT}/acestep-v15-base')
# UI/handler scans <repo>/checkpoints — symlink so model discovery works
dst='/content/ACE-Step-1.5/checkpoints'
if not os.path.islink(dst) and not os.path.exists(dst): os.symlink(CKPT, dst)
import shutil; shutil.copytree(str(DATASET), '/content/data', dirs_exist_ok=True)
# our prep module (from the electronic-lora repo — upload scripts/lm_sft_prep.py to Drive first)
shutil.copy(str(DRIVE/'scripts'/'lm_sft_prep.py'), '/content/lm_sft_prep.py')
print('audio files:', len(list(Path('/content/data/audio').glob('*'))))

In [ ]:
# 4 — Load the base model (gives us the audio->codes encoder)
import sys; sys.path.insert(0, '/content/ACE-Step-1.5')
from acestep.handler import AceStepHandler
h = AceStepHandler()
# config_path = the base model dir; matches what the UI loads as 'base'
status, ok = h.initialize_service(
    project_root='/content/ACE-Step-1.5',
    config_path=f'{CKPT}/acestep-v15-base',
    device='cuda',
    compile_model=False,
)
print(status); assert ok, 'model failed to load'

In [ ]:
# 5 — Round-trip sanity check on ONE chunk BEFORE the full extraction.
#     Confirms the encoder produces codes and the prompt format is sane.
import importlib, lm_sft_prep as P; importlib.reload(P)
import json, tempfile

ds = json.load(open('/content/data/dataset.json'))
sample = ds['samples'][0]
track = f"/content/data/audio/{sample['filename']}"
idx, samples, sr, dur = next(P.iter_chunks(track))
wav = P.write_chunk_wav(samples, sr, tempfile.mktemp(suffix='.wav'))
codes = h.convert_src_audio_to_codes(wav)
print('ERROR' if P.is_error_codes(codes) else 'OK')
print('n code tokens:', codes.count('<|audio_code_'))
print('prompt preview:\n', P.build_prompt(sample['caption'], bpm=sample.get('bpm'), duration=dur)[:300])
print('codes preview:', codes[:120])

In [ ]:
# 6 — Full extraction: 57 tracks -> ~30s chunks -> sft.jsonl (cached on Drive).
#     Skips if the cache already exists. This is the slow, model-bound step.
import json, tempfile, importlib, lm_sft_prep as P; importlib.reload(P)
from pathlib import Path
if SFT_JSONL.exists() and SFT_JSONL.stat().st_size > 0:
    n = sum(1 for _ in open(SFT_JSONL)); print(f'cache hit: {n} examples — skipping extraction')
else:
    ds = json.load(open('/content/data/dataset.json'))
    records = []
    for s in ds['samples']:
        track = f"/content/data/audio/{s['filename']}"
        if not Path(track).exists():
            print('missing:', s['filename']); continue
        for idx, samples, sr, dur in P.iter_chunks(track, chunk_s=30.0, min_s=8.0):
            wav = P.write_chunk_wav(samples, sr, tempfile.mktemp(suffix='.wav'))
            codes = h.convert_src_audio_to_codes(wav)
            Path(wav).unlink(missing_ok=True)
            if P.is_error_codes(codes):
                print('skip', s['filename'], idx, '->', codes[:60]); continue
            prompt = P.build_prompt(s['caption'], bpm=s.get('bpm'),
                                    keyscale=s.get('keyscale'), duration=dur)
            records.append(P.build_record(prompt, codes, source=s['filename'], chunk_index=idx))
        print(f"{s['filename'][:45]:45s} -> {sum(1 for r in records if r['source']==s['filename'])} chunks")
    n = P.write_jsonl(records, SFT_JSONL)
    print(f'wrote {n} SFT examples to {SFT_JSONL}')

In [ ]:
# 7 — Free the DiT/VAE, then load the Qwen3 0.6B LM for SFT.
import gc, torch
del h; gc.collect(); torch.cuda.empty_cache()

import glob
# the 0.6B 5Hz LM was downloaded inside the main repo snapshot
cands = glob.glob(f'{CKPT}/**/acestep-5Hz-lm-0.6B', recursive=True) + \
        glob.glob(f'{CKPT}/**/*5Hz*0.6B*', recursive=True)
LM_PATH = cands[0] if cands else None
print('LM path:', LM_PATH); assert LM_PATH, 'could not find the 0.6B 5Hz LM under checkpoints'

from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(LM_PATH, trust_remote_code=True, use_fast=True)
lm  = AutoModelForCausalLM.from_pretrained(LM_PATH, trust_remote_code=True,
                                           torch_dtype=torch.bfloat16, device_map='cuda')
print('LM loaded:', lm.config.model_type, '| vocab:', len(tok))

In [ ]:
# 8 — Build the HF dataset. Loss on the COMPLETION (codes) only; prompt is masked.
from datasets import load_dataset
raw = load_dataset('json', data_files=str(SFT_JSONL), split='train')
raw = raw.train_test_split(test_size=0.05, seed=42)
print(raw)

RESP_MARK = '\n<|audio_code_'   # completion begins at the first audio code
def to_text(ex):
    # full sequence = prompt + completion + eos; collator masks everything before the codes
    return {'text': ex['prompt'] + ex['completion'] + tok.eos_token}
train_ds = raw['train'].map(to_text, remove_columns=raw['train'].column_names)
eval_ds  = raw['test'].map(to_text,  remove_columns=raw['test'].column_names)
print(train_ds[0]['text'][:400])

In [ ]:
# 9 — LoRA + SFT. Small rank, few epochs: nudge the grooves, don't memorize 57 tracks.
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

peft_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj'],
)
sft_cfg = SFTConfig(
    output_dir=str(LM_OUT),
    per_device_train_batch_size=2, gradient_accumulation_steps=4,
    num_train_epochs=3,                 # start low; watch eval loss for overfit
    learning_rate=1e-4, lr_scheduler_type='cosine', warmup_ratio=0.05,
    logging_steps=10, eval_strategy='epoch', save_strategy='epoch',
    bf16=True, max_seq_length=4096,     # ~30s of 5Hz codes fits comfortably
    report_to='none', seed=42,
)
trainer = SFTTrainer(
    model=lm, args=sft_cfg, train_dataset=train_ds, eval_dataset=eval_ds,
    peft_config=peft_cfg, processing_class=tok,
)
trainer.train()
trainer.save_model(str(LM_OUT / 'final'))
print('saved LM LoRA ->', LM_OUT / 'final')

## Using it

At inference, load **both** adapters: the DiT LoRA (your sound) and this LM LoRA
(your grooves). The LM adapter plugs into the 5Hz planner via PEFT.

## Evaluate

1. Same-seed A/B: LM-LoRA on vs off (DiT LoRA fixed). Listen for arrangement/groove
   shift — fill patterns, phrase pacing, drum placement — not timbre.
2. Watch **eval loss** in cell 9: if it climbs while train loss falls, you're
   overfitting the 57 tracks — drop to 1-2 epochs or lower rank.
3. Compare against DiT-LoRA-only: does adding the LM adapter make the composition
   read as "your tracks" rather than just "your sound"? That's the whole hypothesis.